# Figure 3: MOLLI, ShMOLLI and SASHA sampling

The cell below is tagged `#| label: fig-sampling-output`, so `04-molli-shmolli-sasha.md` embeds its output with `:::{figure} #fig-sampling-output`.

In [ ]:
#| label: fig-sampling-output
# Figure 3: how MOLLI, ShMOLLI and SASHA sample the recovery curve during the heartbeats.
import numpy as np
from scipy.optimize import curve_fit
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# Interactive figure for the website + PNG snapshot for the PDF (same probe as the template).
try:
    pio.to_image(go.Figure(), format="png")
    pio.renderers.default = "plotly_mimetype+png"
except Exception as exc:
    pio.renderers.default = "plotly_mimetype"
    print(f"Static PNG export unavailable ({type(exc).__name__}); PDF will show the caption only.")

# --- Parameters -------------------------------------------------------------
M0 = 1.0
RR = 1000.0            # time between two heartbeats [ms] (60 beats per minute)
ALPHA_EFF = 35.0       # toy model: each single-shot image acts like ONE pulse of this angle
TISSUES = {"myocardium (T1 = 1000 ms)": 1000.0, "blood (T1 = 1600 ms)": 1600.0,
           "myocardium after contrast (T1 = 400 ms)": 400.0}
SEQUENCES = ["MOLLI", "ShMOLLI", "SASHA"]

# --- Timing of each sequence ------------------------------------------------
def schedule(seq):
    """List of events (time [ms], kind). Every image is taken at the same moment of a
    heartbeat, here at t = k * RR. Kinds: 'inv' (inversion), 'sat' (saturation), 'img'."""
    events = []
    if seq in ("MOLLI", "ShMOLLI"):
        # (images after this inversion, rest beats afterwards)
        pattern = [(3, 3), (3, 3), (5, 0)] if seq == "MOLLI" else [(5, 1), (1, 1), (1, 0)]
        beat = 0
        for k, (n_img, n_rest) in enumerate(pattern):
            ti_first = 100 + 80 * k          # shift the TIs a bit for each inversion
            events.append((beat * RR - ti_first, "inv"))
            for j in range(n_img):
                events.append(((beat + j) * RR, "img"))
            beat += n_img + n_rest
    else:  # SASHA: first image without preparation, then one saturation per beat
        ts_values = np.linspace(100, RR - 150, 9)     # saturation times spread over the beat
        events.append((0.0, "img"))
        for j, ts in enumerate(ts_values, start=1):
            events.append((j * RR - ts, "sat"))
            events.append((j * RR, "img"))
    return sorted(events)

# --- Simulation -------------------------------------------------------------
def simulate(seq, T1, dt=5.0):
    """Follow Mz through the whole sequence. Between events Mz recovers with T1.
    Returns the Mz curve and the samples (effective TI or TS, Mz)."""
    events = schedule(seq)
    ca = np.cos(np.deg2rad(ALPHA_EFF))
    t, mz, last_prep = events[0][0] - 300, M0, None
    t_curve, mz_curve, samples = [t], [mz], []
    for t_ev, kind in events:
        tt = np.arange(t, t_ev, dt)                          # recovery until the event
        t_curve += list(tt); mz_curve += list(M0 - (M0 - mz) * np.exp(-(tt - t) / T1))
        mz, t = M0 - (M0 - mz) * np.exp(-(t_ev - t) / T1), t_ev
        t_curve.append(t); mz_curve.append(mz)
        if kind == "inv":
            mz, last_prep = -mz, t           # perfect inversion
        elif kind == "sat":
            mz, last_prep = 0.0, t           # perfect saturation: history erased
        else:
            eff = t - last_prep if last_prep is not None else np.inf
            samples.append((eff, mz))
            mz = mz * ca                     # the readout costs a bit of Mz
        t_curve.append(t); mz_curve.append(mz)
    tt = np.arange(t, t + 1500, dt)
    t_curve += list(tt); mz_curve += list(M0 - (M0 - mz) * np.exp(-(tt - t) / T1))
    return np.array(t_curve), np.array(mz_curve), events, samples

def fit(seq, samples):
    x = np.array([s[0] for s in samples]); y = np.array([s[1] for s in samples])
    if seq == "SASHA":
        x = np.where(np.isinf(x), 1e5, x)    # first image: no saturation, fully recovered
        model = lambda ts, A, eta, T1: A * (1 - eta * np.exp(-ts / T1))
        (A, eta, T1), _ = curve_fit(model, x, y, p0=[1, 1, 1000], maxfev=20000)
        return model, (A, eta, T1), f"fit: T1 = {T1:.0f} ms", len(x)
    model = lambda ti, A, B, T1s: A - B * np.exp(-ti / T1s)
    n_used = len(x)
    if seq == "ShMOLLI":
        # Simple rule for this book: fit the first 5 images; keep images 6 and 7 only if
        # that T1 is shorter than half an RR interval (enough recovery in between).
        (A, B, T1s), _ = curve_fit(model, x[:5], y[:5], p0=[1, 2, 1000], maxfev=20000)
        n_used = 7 if T1s * (B / A - 1) < RR / 2 else 5
    (A, B, T1s), _ = curve_fit(model, x[:n_used], y[:n_used], p0=[1, 2, 1000], maxfev=20000)
    return model, (A, B, T1s), f"fit: T1* = {T1s:.0f} ms, corrected T1 = {T1s * (B / A - 1):.0f} ms", n_used

# --- Figure -----------------------------------------------------------------
fig = make_subplots(rows=2, cols=1, vertical_spacing=0.17,
                    subplot_titles=("What happens during the scan",
                                    "All images sorted by their time after the preparation pulse"))
COLORS = {"img": "black", "inv": "royalblue", "sat": "darkorange"}
options, TRACES = [], 6
for seq in SEQUENCES:
    for tissue, T1 in TISSUES.items():
        options.append((seq, tissue))
        vis = (seq == "MOLLI" and T1 == 1000.0)
        t_c, mz_c, events, samples = simulate(seq, T1)
        t0 = t_c[0]
        # ECG: one R-peak marker per heartbeat
        beats = np.arange(np.ceil(t0 / RR) * RR, t_c[-1], RR)
        ecg_x, ecg_y = [], []
        for b in beats:
            ecg_x += [b - 40, b, b + 40, None]; ecg_y += [-1.35, -1.12, -1.35, None]
        fig.add_trace(go.Scatter(x=(np.array(ecg_x, dtype=float) - t0) / 1000, y=ecg_y,
                                 mode="lines", line=dict(color="gray"), name="heartbeat (ECG)",
                                 visible=vis), row=1, col=1)
        fig.add_trace(go.Scatter(x=(t_c - t0) / 1000, y=mz_c, mode="lines",
                                 line=dict(color="lightgray", width=3), name="Mz",
                                 visible=vis), row=1, col=1)
        prep = [e for e in events if e[1] != "img"]
        px, py = [], []
        for t_ev, _ in prep:
            px += [(t_ev - t0) / 1000] * 2 + [None]; py += [-1.1, 1.1, None]
        kind = "inv" if seq != "SASHA" else "sat"
        fig.add_trace(go.Scatter(x=px, y=py, mode="lines",
                                 line=dict(color=COLORS[kind], dash="dot"),
                                 name="inversion pulse" if kind == "inv" else "saturation pulse",
                                 visible=vis), row=1, col=1)
        img_t = [(e[0] - t0) / 1000 for e in events if e[1] == "img"]
        fig.add_trace(go.Scatter(x=img_t, y=[s[1] for s in samples], mode="markers",
                                 marker=dict(size=9, color="black"), name="image",
                                 visible=vis), row=1, col=1)
        # bottom: samples sorted by effective time, plus the fit
        model, p, label, n_used = fit(seq, samples)
        x = np.array([s[0] for s in samples])
        x_plot = np.where(np.isinf(x), 4400, x)        # SASHA's first image drawn at the right edge
        colors = ["black"] * n_used + ["darkorange"] * (len(x) - n_used)
        fig.add_trace(go.Scatter(x=x_plot, y=[s[1] for s in samples], mode="markers",
                                 marker=dict(size=9, color=colors),
                                 name="images (orange: not used in the fit)" if n_used < len(x) else "images",
                                 visible=vis), row=2, col=1)
        xf = np.linspace(0, 4500, 300)
        fig.add_trace(go.Scatter(x=xf, y=model(xf, *p), mode="lines",
                                 line=dict(color="crimson", dash="dash"),
                                 name=label + f" (true T1 = {T1:.0f} ms)", visible=vis), row=2, col=1)

buttons = []
for i, (seq, tissue) in enumerate(options):
    visible = [False] * (TRACES * len(options))
    visible[i * TRACES:(i + 1) * TRACES] = [True] * TRACES
    buttons.append(dict(label=f"{seq}, {tissue}", method="update", args=[{"visible": visible}]))

fig.update_layout(
    updatemenus=[dict(buttons=buttons, direction="down", x=0, xanchor="left", y=1.18, yanchor="top")],
    height=750, legend=dict(orientation="h", y=-0.15), margin=dict(t=110),
    template="plotly_white",
)
fig.update_xaxes(title_text="Time [s]", row=1, col=1)
fig.update_yaxes(title_text="Mz / M0", range=[-1.4, 1.15], row=1, col=1)
fig.update_xaxes(title_text="Time after inversion (TI) or saturation (TS) [ms]", range=[0, 4600], row=2, col=1)
fig.update_yaxes(title_text="Mz / M0", range=[-1.1, 1.1], row=2, col=1)
fig
